In [2]:
import os
import sys
import json
import pandas as pd 
import networkx as nx 
import pickle as pkl

import datetime
import pytz

import re

In [3]:
# get directory list \n",
myFiles = []
def absoluteFilePaths(directory):
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if ".json" in f:
                myFiles.append(os.path.abspath(os.path.join(dirpath, f)))

absoluteFilePaths("./")
myFiles = sorted(myFiles)
for file in myFiles:
    print(file)

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-001-025/AIA-1-25.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-051-075/AIA-51-75.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-101-125/AIA-101-125.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-2019-12-08T22-39-56.146.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-201-225/AIA-201-225.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-251-275/AIA-251-275.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-301-325/AIA-301-325.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-351-375/AIA-351-375.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-401-425/AIA-401-425.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-451-475/AIA-451-475.

In [4]:
dt_beg = datetime.datetime(2019, 9, 25, 9, 0, 0, tzinfo=pytz.timezone("Etc/GMT+4"))
dt_end = datetime.datetime(2019, 9, 25, 14, 40, 0, tzinfo=pytz.timezone("Etc/GMT+4"))
tdelta = datetime.timedelta(minutes = 10)

tslots = []
stime = dt_beg
etime = dt_beg + tdelta
tslots.append((stime, etime))
                                                                           
while etime < dt_end:
    stime = stime + tdelta
    etime = etime + tdelta
    if etime > dt_end:
        etime = dt_end
    tslots.append((stime, etime))

for dt in tslots:
    print(dt[0].isoformat(), dt[1].isoformat())

2019-09-25T09:00:00-04:00 2019-09-25T09:10:00-04:00
2019-09-25T09:10:00-04:00 2019-09-25T09:20:00-04:00
2019-09-25T09:20:00-04:00 2019-09-25T09:30:00-04:00
2019-09-25T09:30:00-04:00 2019-09-25T09:40:00-04:00
2019-09-25T09:40:00-04:00 2019-09-25T09:50:00-04:00
2019-09-25T09:50:00-04:00 2019-09-25T10:00:00-04:00
2019-09-25T10:00:00-04:00 2019-09-25T10:10:00-04:00
2019-09-25T10:10:00-04:00 2019-09-25T10:20:00-04:00
2019-09-25T10:20:00-04:00 2019-09-25T10:30:00-04:00
2019-09-25T10:30:00-04:00 2019-09-25T10:40:00-04:00
2019-09-25T10:40:00-04:00 2019-09-25T10:50:00-04:00
2019-09-25T10:50:00-04:00 2019-09-25T11:00:00-04:00
2019-09-25T11:00:00-04:00 2019-09-25T11:10:00-04:00
2019-09-25T11:10:00-04:00 2019-09-25T11:20:00-04:00
2019-09-25T11:20:00-04:00 2019-09-25T11:30:00-04:00
2019-09-25T11:30:00-04:00 2019-09-25T11:40:00-04:00
2019-09-25T11:40:00-04:00 2019-09-25T11:50:00-04:00
2019-09-25T11:50:00-04:00 2019-09-25T12:00:00-04:00
2019-09-25T12:00:00-04:00 2019-09-25T12:10:00-04:00
2019-09-25T1

In [5]:
def get_timeslot(ts):
    for i in range(len(tslots)):
        interval = tslots[i]
        if ts > interval[0] and ts <= interval[1]:
            return i

t = datetime.datetime.fromisoformat("2019-09-25T09:12:00.000-04:00")
print(get_timeslot(t))

1


In [ ]:
# 1 Select the graph 
# 2 read it from file 
# 3 update it 
# write it back 

In [10]:
# read the graph
gIdx = 2
G = nx.MultiDiGraph()
dir_string = "graph_{}".format(gIdx)
G = nx.read_gpickle(dir_string)

for fileName in myFiles[2:]:
    print(fileName)
    with open(fileName) as logFile:
        for line in logFile:
            line = line.strip()
            try:
                y = json.loads(line)
            except:
                print(line)
                
            try:
                timestamp = re.sub(r'\.\d+', '', y["timestamp"])
                ts = datetime.datetime.fromisoformat(timestamp)
                idx = get_timeslot(ts)
            except:
                print(y["timestamp"])
                continue
            
            if idx < gIdx:
                continue
            elif idx > gIdx:
                break
            else:
                myHost = y["hostname"]
                actorId = y["actorID"]
                objectId = y["objectID"]
                objectType = y["object"]

                G.add_node(actorId)
                G.add_node(objectId)

                G.nodes[actorId]["pid"] = y["pid"]
                G.nodes[actorId]["oType"] = "PROCESS"
                G.nodes[objectId]["oType"] = objectType

                properties = y["properties"]
                properties["action"] = y["action"]
                properties["principal"] = y["principal"]
                properties["timestamp"] = y["timestamp"]

                #[key = value for key,value in enum(properties)]\n",
                G.add_edge(y["actorID"], y["objectID"], attributes = properties)
nx.write_gpickle(G, dir_string)

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-101-125/AIA-101-125.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-2019-12-08T22-39-56.146.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-201-225/AIA-201-225.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-251-275/AIA-251-275.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-301-325/AIA-301-325.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-351-375/AIA-351-375.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-401-425/AIA-401-425.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-451-475/AIA-451-475.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-501-525/AIA-501-525.ecar-2019-11-17T15-04-02.073.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/

In [11]:
print(G.number_of_nodes())
print(G.number_of_edges())

2061520
13331628


In [ ]:
for gIdx in range(20, len(tslots)):
    # read the graph
    #gIdx = 1
    G = nx.MultiDiGraph()
    dir_string = "graph_{}".format(gIdx)
    G = nx.read_gpickle(dir_string)

    print("Graph: {}".format(dir_string))
    # Update the graph
    for fileName in myFiles[2:]:
        print(fileName)
        with open(fileName) as logFile:
            for line in logFile:
                line = line.strip()
                try:
                    y = json.loads(line)
                except:
                    print(line)

                try:
                    timestamp = re.sub(r'\.\d+', '', y["timestamp"])
                    ts = datetime.datetime.fromisoformat(timestamp)
                    idx = get_timeslot(ts)
                except:
                    print(y["timestamp"])
                    continue
                
                if idx < gIdx:
                    continue
                elif idx > gIdx:
                    break
                else:
                    myHost = y["hostname"]
                    actorId = y["actorID"]
                    objectId = y["objectID"]
                    objectType = y["object"]

                    G.add_node(actorId)
                    G.add_node(objectId)

                    G.nodes[actorId]["pid"] = y["pid"]
                    G.nodes[actorId]["oType"] = "PROCESS"
                    G.nodes[objectId]["oType"] = objectType

                    properties = y["properties"]
                    properties["action"] = y["action"]
                    properties["principal"] = y["principal"]
                    properties["timestamp"] = y["timestamp"]

                    #[key = value for key,value in enum(properties)]\n",
                    G.add_edge(y["actorID"], y["objectID"], attributes = properties)
    # Write the graph 
    nx.write_gpickle(G, dir_string)

Graph: graph_20
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-101-125/AIA-101-125.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-2019-12-08T22-39-56.146.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-201-225/AIA-201-225.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-251-275/AIA-251-275.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-301-325/AIA-301-325.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-351-375/AIA-351-375.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-401-425/AIA-401-425.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-451-475/AIA-451-475.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-501-525/AIA-501-525.ecar-2019-11-17T15-04-02.073.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/ev

2019-09-25T12:27:26.906-04:00
2019-09-25T12:27:39.154-04:00
2019-09-25T12:28:26.393-04:00
{"action":"MESSAGE","actorID":"0738866c-abaa-40a8-af60-0a343eb4cb20","hostname":"SysClient0116.systemia.com","id":"0fa0c73a-b351-4629-9b14-ea7bd18a72e5","object":"FLOW","objectID":"28832957-c66e-4977-ae6a-5c00e02fd293","pid":912,"ppid":552,"principal":"NT AUTHORITY\\NETWORK SERVICE","properties":{"acuity_level":"4","dest_ip":"224.0.0.252","dest_port":"5355","direction":"inbound","image_path":"\\Device\\HarddiskVolume1\\Windows\\system32\\svchost.exe","l4protocol":"17","size":"23","src_ip":"10.20.2.28","src_port":"54468"},"tid":-1,"timestamp":"2019-09-25T12:28:39.909-04:00"}
2019-09-25T12:29:04.544-04:00
{"action":"MESSAGE","actorID":"dad82247-0ae1-4442-af59-024a82e811ad","hostname":"SysClient0122.systemia.com","id":"72a8f788-2263-4e17-98a4-967510aea561","object":"FLOW","objectID":"3b7848b2-19cd-4168-ad2f-c06b6ddff1b4","pid":4,"ppid":0,"principal":"NT AUTHORITY\\SYSTEM","properties":{"acuity_level"

In [ ]:
for gIdx in range(len(tslots), 22, -1):
    # read the graph
    #gIdx = 1
    G = nx.MultiDiGraph()
    dir_string = "graph_{}".format(gIdx)
    #G = nx.read_gpickle(dir_string)

    print("Graph: {}".format(dir_string))
    
    # Update the graph
    for fileName in myFiles[2:]:
        print(fileName)
        f = open(fileName, "r")
        logFile = f.readlines()
        f.close()

        logFile.reverse()
        for line in logFile:
            line = line.strip()
            try:
                y = json.loads(line)
            except:
                print(line)

            try:
                timestamp = re.sub(r'\.\d+', '', y["timestamp"])
                ts = datetime.datetime.fromisoformat(timestamp)
                idx = get_timeslot(ts)
            except:
                print(y["timestamp"])
                continue

            if idx > gIdx:
                continue
            elif idx < gIdx:
                break
            else:
                myHost = y["hostname"]
                actorId = y["actorID"]
                objectId = y["objectID"]
                objectType = y["object"]

                G.add_node(actorId)
                G.add_node(objectId)

                G.nodes[actorId]["pid"] = y["pid"]
                G.nodes[actorId]["oType"] = "PROCESS"
                G.nodes[objectId]["oType"] = objectType

                properties = y["properties"]
                properties["action"] = y["action"]
                properties["principal"] = y["principal"]
                properties["timestamp"] = y["timestamp"]

                #[key = value for key,value in enum(properties)]\n",
                G.add_edge(y["actorID"], y["objectID"], attributes = properties)
    # Write the graph 
    nx.write_gpickle(G, dir_string)

Graph: graph_34
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-101-125/AIA-101-125.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-2019-12-08T22-39-56.146.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-151-175/AIA-151-175.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-201-225/AIA-201-225.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-251-275/AIA-251-275.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-301-325/AIA-301-325.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-351-375/AIA-351-375.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-401-425/AIA-401-425.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-451-475/AIA-451-475.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-501-525/AIA-501-525.ecar-2019-11-17T15-04-02.073.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/ev

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-201-225/AIA-201-225.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-251-275/AIA-251-275.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-301-325/AIA-301-325.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-351-375/AIA-351-375.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-401-425/AIA-401-425.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-451-475/AIA-451-475.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-501-525/AIA-501-525.ecar-2019-11-17T15-04-02.073.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-501-525/AIA-501-525.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-551-575/AIA-551-575.ecar-2019-11-17T15-01-05.424.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-551-575/AIA-551-575.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-351-375/AIA-351-375.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-401-425/AIA-401-425.ecar-last.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-451-475/AIA-451-475.ecar-last.json
